# Session 20 Exercise: Regression Trees

Compare shallow and deeper regression trees for housing prices.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
X = housing[features]
y = housing["price_k_eur"]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["district"]),
        ("num", "passthrough", ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]),
    ]
)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
rows = []
for depth in [2, 3, 4, 6, None]:
    model = Pipeline(
        [
            ("preprocess", preprocess),
            ("model", DecisionTreeRegressor(max_depth=depth, min_samples_leaf=10, random_state=42)),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"max_depth": depth, "test_rmse": mean_squared_error(y_test, pred) ** 0.5})
pd.DataFrame(rows)


Which depth would you choose? Explain the choice in terms of error and interpretability.
